In [2]:
%matplotlib inline

In [3]:
import phoenix as px
import os
import json
from tqdm import tqdm
from phoenix.trace import SpanEvaluations
from phoenix.trace.dsl import SpanQuery
from openinference.instrumentation import suppress_tracing

# to run multiple calls asynchronously for faster evaluations.
import nest_asyncio
nest_asyncio.apply()

c:\Users\Nada\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PROJECT_NAME = "evaluating-agent"
from openai import OpenAI
import pandas as pd
import json
import duckdb
from pydantic import BaseModel, Field
from IPython.display import Markdown


import phoenix as px
import os
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.trace import Status, StatusCode
from openinference.instrumentation import TracerProvider
from phoenix.otel import register
from helper import get_openai_api_key, get_phoenix_trace_endpoint

# initialize the OpenAI client
openai_api_key = get_openai_api_key()
client = OpenAI(api_key=openai_api_key)
MODEL = "gpt-4o-mini"

PROJECT_NAME = "evaluating-agent"
tracer_provider = register(
    project_name=PROJECT_NAME,
    endpoint=get_phoenix_trace_endpoint(),
    protocol="http/protobuf",
)

OpenAIInstrumentor().instrument(tracer_provider = tracer_provider)
tracer = tracer_provider.get_tracer(__name__)


# Defining the tools

# Tool 1: Database Lookup
TRANSACTION_DATA_FILE_PATH = './data/Store_Sales_Price_Elasticity_Promotions_Data.parquet'

# prompt template for step 2 of tool 1
SQL_GENERATION_PROMPT = """
Generate an SQL query based on a prompt. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

# code for step 2 of tool 1
def generate_sql_query(prompt: str, columns: list, table_name: str) -> str:
    """Generate an SQL query based on a prompt"""
    formatted_prompt = SQL_GENERATION_PROMPT.format(prompt=prompt, columns=columns, table_name=table_name)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    return response.choices[0].message.content


# code for tool 1
@tracer.tool()
def lookup_sales_data(prompt: str) -> str:
    """Implementation of sales data lookup from parquet file using SQL"""
    try:

        # define the table name
        table_name = "sales"
        
        # step 1: read the parquet file into a DuckDB table
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")

        # step 2: generate the SQL code
        sql_query = generate_sql_query(prompt, df.columns, table_name)
        # clean the response to make sure it only includes the SQL code
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")
        
        # step 3: execute the SQL query

        with tracer.start_as_current_span("execute_sql_query", openinference_span_kind="chain") as span:
            
            result = duckdb.sql(sql_query).df()

            span.set_output(value=str(result))
            span.set_status(StatusCode.OK)
        
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {str(e)}"


#### Tool 2: Data Analysis


# Construct prompt based on analysis type and data subset
DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""

# code for tool 2
@tracer.tool()
def analyze_sales_data(prompt: str, data: str) -> str:
    """Implementation of AI-powered sales data analysis"""
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated"


# ### Tool 3: Data Visualization
    
# prompt template for step 1 of tool 3
CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""

# class defining the response format of step 1 of tool 3
class VisualizationConfig(BaseModel):
    chart_type: str = Field(..., description="Type of chart to generate")
    x_axis: str = Field(..., description="Name of the x-axis column")
    y_axis: str = Field(..., description="Name of the y-axis column")
    title: str = Field(..., description="Title of the chart")


# code for step 1 of tool 3
@tracer.chain()
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """Generate chart visualization configuration
    
    Args:
        data: String containing the data to visualize
        visualization_goal: Description of what the visualization should show
        
    Returns:
        Dictionary containing line chart configuration
    """
    formatted_prompt = CHART_CONFIGURATION_PROMPT.format(data=data, visualization_goal=visualization_goal)
    
    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
        response_format=VisualizationConfig,
    )
    
    try:
        # Extract axis and title info from response
        content = response.choices[0].message.content
        
        # Return structured chart config
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data
        }
    except Exception:
        return {
            "chart_type": "line", 
            "x_axis": "date",
            "y_axis": "value",
            "title": visualization_goal,
            "data": data
        }



CREATE_CHART_PROMPT = """
Write python code to create a chart based on the following configuration.
Only return the code, no other text.
config: {config}
"""


# code for step 2 of tool 3
@tracer.chain()
def create_chart(config: dict) -> str:
    """Create a chart based on the configuration"""
    formatted_prompt = CREATE_CHART_PROMPT.format(config=config)
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    
    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "")
    code = code.strip()
    
    return code

    
# code for tool 3
@tracer.tool()
def generate_visualization(data: str, visualization_goal: str) -> str:
    """Generate a visualization based on the data and goal"""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code


# ## Tool Schema

    
# Define tools/functions that can be called by the model
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up data from Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data", 
            "description": "Analyze sales data to extract insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate Python code to create data visualizations",
            "parameters": {
                "type": "object", 
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "visualization_goal": {"type": "string", "description": "The goal of the visualization."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

# Dictionary mapping function names to their implementations
tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data, 
    "generate_visualization": generate_visualization
}



# code for executing the tools returned in the model's response
@tracer.chain()
def handle_tool_calls(tool_calls, messages):
    
    for tool_call in tool_calls:
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)
       
        messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        
    return messages




SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""




def run_agent(messages):
    print("Running agent with messages:", messages)
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not any(
            isinstance(message, dict) and message.get("role") == "system" for message in messages
        ):
            system_prompt = {"role": "system", "content": SYSTEM_PROMPT}
            messages.append(system_prompt)

    while True:
        # Router Span
        print("Starting router call span")
        with tracer.start_as_current_span(
            "router_call",
            openinference_span_kind="chain",
        ) as span:
            span.set_input(value=messages)
            
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
            )
            messages.append(response.choices[0].message.model_dump())
            tool_calls = response.choices[0].message.tool_calls
            print("Received response with tool calls:", bool(tool_calls))
            span.set_status(StatusCode.OK)
    
            if tool_calls:
                print("Starting tool calls span")
                messages = handle_tool_calls(tool_calls, messages)
                span.set_output(value=tool_calls)
            else:
                print("No tool calls, returning final response")
                span.set_output(value=response.choices[0].message.content)
                return response.choices[0].message.content


### Creating the Main Span
def start_main_span(messages):
    print("Starting main span with messages:", messages)

    # This is manual tracing call.
    with tracer.start_as_current_span(
        "AgentRun", openinference_span_kind="agent"
    ) as span:
        span.set_input(value=messages)
        # Call the run_agent function to execute the agent's logic
        ret = run_agent(messages)
        print("Main span completed with return value:", ret)
        span.set_output(value=ret)
        span.set_status(StatusCode.OK)
        return ret




Phoenix endpoint: http://localhost:6006/
OpenTelemetry Tracing Details
|  Phoenix Project: evaluating-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Running Agent with a Set of Testing Questions
To evaluate your agent's components, you will run the agent using a set of questions. For each question, you will collect spans and send them to Phoenix. Next to evaluate an agent component, you will query some specific spans and use them as your testing examples for your evaluators. Finally, you will upload the evaluated spans to Phoenix.

<img src="./images/traces.png" width="400"/>

In [5]:
agent_questions = [
    "What was the most popular product SKU?",
    "What was the total revenue across all stores?",
    "Which store had the highest sales volume?",
    "Create a bar chart showing total sales by store",
    "What percentage of items were sold on promotion?",
    "What was the average transaction value?"
]

for question in tqdm(agent_questions, desc="Processing questions"):
    try:
        ret = start_main_span([{"role": "user", "content": question}])
    except Exception as e:
        print(f"Error processing question: {question}")
        print(e)
        continue

Processing questions:   0%|          | 0/6 [00:00<?, ?it/s]

Starting main span with messages: [{'role': 'user', 'content': 'What was the most popular product SKU?'}]
Running agent with messages: [{'role': 'user', 'content': 'What was the most popular product SKU?'}]
Starting router call span
Received response with tool calls: True
Starting tool calls span
Starting router call span


Processing questions:  17%|█▋        | 1/6 [00:07<00:37,  7.48s/it]

Received response with tool calls: False
No tool calls, returning final response
Main span completed with return value: The most popular product SKU was **6200700**, with a total quantity sold of **52,262**.
Starting main span with messages: [{'role': 'user', 'content': 'What was the total revenue across all stores?'}]
Running agent with messages: [{'role': 'user', 'content': 'What was the total revenue across all stores?'}]
Starting router call span
Received response with tool calls: True
Starting tool calls span
Starting router call span


Processing questions:  33%|███▎      | 2/6 [00:10<00:18,  4.68s/it]

...

Received response with tool calls: False
No tool calls, returning final response
Main span completed with return value: Here is the bar chart showing total sales by store:

```python
import pandas as pd
import matplotlib.pyplot as plt

data = {
    'Store_Number': [880, 1650, 4180, 550, 1100, 3300, 3190, 2970, 3740, 2530, 
                     4400, 3630, 2310, 2200, 1870, 4730, 4070, 3080, 2090, 1320, 
                     2640, 1540, 4840, 3410, 990, 1210, 330, 2750, 1980, 1760, 
                     2860, 2420, 770, 3520, 660],
    'Total_Sales': [420302.088397, 580443.007953, 272208.118542, 229727.498752,
                    497509.528013, 619660.167018, 335035.018792, 836341.327191,
                    359729.808228, 324046.518720, 95745.620250, 405034.547846,
                    412579.388504, 361173.288199, 401070.997685, 239711.708869,
                    322307.968330, 495458.238811, 309996.247965, 592832.067579,
                    308990.318559, 427777.427815, 389056.668316,

Processing questions:  83%|████████▎ | 5/6 [00:49<00:11, 11.76s/it]

Received response with tool calls: False
No tool calls, returning final response
Main span completed with return value: It appears that the data access is consistently encountering a conversion error with the on-promotion status. Unfortunately, this means I cannot retrieve the required sales data directly at the moment.

Would you like me to assist you in another way, perhaps by providing an alternative analysis or a different question related to the dataset?
Starting main span with messages: [{'role': 'user', 'content': 'What was the average transaction value?'}]
Running agent with messages: [{'role': 'user', 'content': 'What was the average transaction value?'}]
Starting router call span
Received response with tool calls: True
Starting tool calls span
Starting router call span


Processing questions: 100%|██████████| 6/6 [00:52<00:00,  8.67s/it]

Received response with tool calls: False
No tool calls, returning final response
Main span completed with return value: The average transaction value was approximately $19.02.


## Router Evals using LLM-as-a-Judge
To evaluate the router, you will use this template provided by Phoenix to the LLM-as-a-Judge.
To create evaluations in phoenix we should export the spans that we created, then perform evaluations on them and filter through them to reach the spans that we want to evaluate.

In [6]:
query = SpanQuery().where(
    # Filter for the `LLM` span kind.
    # The filter condition is a string of valid Python boolean expression.
    "span_kind == 'LLM'", # Gives us all the spans that are of kind `LLM`.
).select( # select the attributes we want to see in the resulting dataframe.
    # In this case, we want to see the question that was asked and the tool call that was made.
    # These exact values you can get from Phoenix dashbaoard, select the chatcompletion span (LLM call)
    question="input.value",
    tool_call="llm.tools"
)

# Query spans with the Phoenix client API available in the installed SDK.
from phoenix.client import Client as PhoenixClient

tool_calls_df = PhoenixClient().spans.get_spans_dataframe(
    query=query,
    project_name=PROJECT_NAME,
    timeout=None,
)
# drop all spans that don't have a tool call, as we only want to see the spans that have a tool call.
tool_calls_df = tool_calls_df.dropna(subset=["tool_call"])

tool_calls_df.head()

,question,tool_call
context.span_id,,
4c526afe6a5cb163,"{""model"": ""gpt-4o-mini"", ""messages"": [{""role"":...","[{'tool': {'json_schema': '{""type"": ""function""..."
2cd0a7068668c601,"{""model"": ""gpt-4o-mini"", ""messages"": [{""role"":...","[{'tool': {'json_schema': '{""type"": ""function""..."
bcd6c67ef351b42f,"{""model"": ""gpt-4o-mini"", ""messages"": [{""role"":...","[{'tool': {'json_schema': '{""type"": ""function""..."
d024954bffa657ce,"{""model"": ""gpt-4o-mini"", ""messages"": [{""role"":...","[{'tool': {'json_schema': '{""type"": ""function""..."
0a3dfb25d45b5845,"{""model"": ""gpt-4o-mini"", ""messages"": [{""role"":...","[{'tool': {'json_schema': '{""type"": ""function""..."


In [7]:
from phoenix.evals import LLM, ClassificationEvaluator, evaluate_dataframe
from PROMPTS import TOOL_CALLING_PROMPT

llm_judge = LLM(
    provider="openai",
    model=MODEL,
    api_key=openai_api_key,
)

tool_call_eval = ClassificationEvaluator(
    name="Tool Calling Eval",
    prompt_template=TOOL_CALLING_PROMPT.replace(
        "{tool_definitions}",
        json.dumps(tools).replace("{", '\"').replace("}", '\"'),
    ),
    choices=["correct", "incorrect"],
    llm=llm_judge,
    include_explanation=True,
)

evaluation_results = evaluate_dataframe(
    dataframe=tool_calls_df,
    evaluators=[tool_call_eval],
)
print(evaluation_results.head())


Evaluating Dataframe |██████████| 31/31 (100.0%) | ⏳ 00:43<00:00 |  1.40s/it

                                                           question  \
context.span_id                                                       
4c526afe6a5cb163  {"model": "gpt-4o-mini", "messages": [{"role":...   
2cd0a7068668c601  {"model": "gpt-4o-mini", "messages": [{"role":...   
bcd6c67ef351b42f  {"model": "gpt-4o-mini", "messages": [{"role":...   
d024954bffa657ce  {"model": "gpt-4o-mini", "messages": [{"role":...   
0a3dfb25d45b5845  {"model": "gpt-4o-mini", "messages": [{"role":...   

                                                          tool_call  \
context.span_id                                                       
4c526afe6a5cb163  [{'tool': {'json_schema': '{"type": "function"...   
2cd0a7068668c601  [{'tool': {'json_schema': '{"type": "function"...   
bcd6c67ef351b42f  [{'tool': {'json_schema': '{"type": "function"...   
d024954bffa657ce  [{'tool': {'json_schema': '{"type": "function"...   
0a3dfb25d45b5845  [{'tool': {'json_schema': '{"type": "function"...   

    

In [8]:
# Let's log the evaluation results to Phoenix for further analysis and visualization. We can use the `to_annotation_dataframe` utility to convert the evaluation results into a format suitable for logging as annotations.
from phoenix.evals.utils import to_annotation_dataframe
# Format evaluation results for logging
annotations_df = to_annotation_dataframe(evaluation_results)
print(annotations_df)


C:\Users\Nada\AppData\Local\Temp\ipykernel_28508\690745285.py:4: DeprecationWarning: Positional arguments for to_annotation_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  annotations_df = to_annotation_dataframe(evaluation_results)


     context.span_id  score      label  \
0   4c526afe6a5cb163    NaN    correct   
1   2cd0a7068668c601    NaN    correct   
2   bcd6c67ef351b42f    NaN  incorrect   
3   d024954bffa657ce    NaN    correct   
4   0a3dfb25d45b5845    NaN  incorrect   
5   a40e083b946b710c    NaN    correct   
6   ded369e35d0dc656    NaN    correct   
7   1042707e1e51bdb4    NaN    correct   
8   cffa3eeabaa03899    NaN  incorrect   
9   7854c03748027c74    NaN    correct   
10  65ee88b5d9fc9d54    NaN    correct   
11  ebd9f3c702b785dc    NaN    correct   
12  2091a7e311b3042e    NaN    correct   
13  5152d87b4d8dbc13    NaN    correct   
14  02ac5ccbf0f0e5d9    NaN  incorrect   
15  43d798acd3acf5aa    NaN    correct   
16  a3dcca1ae8ac858e    NaN    correct   
17  00979d69eb192998    NaN    correct   
18  dc9c77f1c869b321    NaN  incorrect   
19  25455749a75117ba    NaN    correct   
20  6542a569649d42b2    NaN    correct   
21  2a760ff43db508ee    NaN    correct   
22  e45fd780beec1d25    NaN    cor

In [9]:
# column score is Nan, so map the label to a score of 1 for correct and 0 for incorrect
annotations_df['score'] = annotations_df.apply(lambda x: 1 if x['label']=='correct' else 0, axis=1)
# print first 5 rows of the annotations dataframe to see it
print(annotations_df.head())


    context.span_id  score      label  \
0  4c526afe6a5cb163      1    correct   
1  2cd0a7068668c601      1    correct   
2  bcd6c67ef351b42f      0  incorrect   
3  d024954bffa657ce      1    correct   
4  0a3dfb25d45b5845      0  incorrect   

                                         explanation  \
0  The chosen tool calls include sequential calls...   
1  The chosen tool call, `lookup_sales_data`, is ...   
2  The tools called are appropriate for analyzing...   
3  The chosen tool 'lookup_sales_data' is appropr...   
4  The assistant correctly called the "lookup_sal...   

                                            metadata    annotation_name  \
0  {'model': 'gpt-4o-mini', 'trace_id': 'dfb2cdf9...  Tool Calling Eval   
1  {'model': 'gpt-4o-mini', 'trace_id': 'bcef1fc0...  Tool Calling Eval   
2  {'model': 'gpt-4o-mini', 'trace_id': 'faa6bca1...  Tool Calling Eval   
3  {'model': 'gpt-4o-mini', 'trace_id': 'febef93e...  Tool Calling Eval   
4  {'model': 'gpt-4o-mini', 'trace_id': '

In [10]:
PhoenixClient().spans.log_span_annotations_dataframe(
    dataframe=annotations_df,
    annotator_kind="LLM",
    annotation_name="Tool Calling Eval"
)
